# Boolean Retrieval: Indices & Inverted Index

Notebook ini mendemonstrasikan pembuatan **inverted index** dan eksekusi **boolean query** (AND, OR, NOT) pada koleksi dokumen kecil berbahasa Indonesia.

**Alur:**
1. Definisi koleksi dokumen.
2. Preprocessing: case folding, tokenisasi, stopword removal (NLTK), stemming (Sastrawi).
3. Pembangunan inverted index (term → posting list).
4. Eksekusi boolean query: AND, OR, NOT.

## 1. Import Library & Inisialisasi

In [1]:
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords', quiet=True)
stop_id = set(stopwords.words('indonesian'))
stemmer = StemmerFactory().create_stemmer()
print('Jumlah stopword Indonesia (NLTK):', len(stop_id))

Jumlah stopword Indonesia (NLTK): 757


## 2. Koleksi Dokumen

Empat dokumen sederhana sebagai contoh.

In [2]:
documents = {
    'D1': 'Mahasiswa belajar sistem temu kembali informasi.',
    'D2': 'Sistem informasi mengelola data mahasiswa di kampus.',
    'D3': 'Temu kembali dokumen menggunakan inverted index.',
    'D4': 'Mahasiswa mengakses dokumen melalui sistem temu kembali.',
}
for doc_id, text in documents.items():
    print(f'{doc_id}: {text}')

D1: Mahasiswa belajar sistem temu kembali informasi.
D2: Sistem informasi mengelola data mahasiswa di kampus.
D3: Temu kembali dokumen menggunakan inverted index.
D4: Mahasiswa mengakses dokumen melalui sistem temu kembali.


## 3. Preprocessing

Tahapan: lowercase → hapus tanda baca → tokenisasi → buang stopword → stemming.

In [3]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_id]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

tokenized = {doc_id: preprocess(text) for doc_id, text in documents.items()}
for doc_id, tokens in tokenized.items():
    print(f'{doc_id}: {tokens}')

D1: ['mahasiswa', 'ajar', 'sistem', 'temu', 'informasi']
D2: ['sistem', 'informasi', 'kelola', 'data', 'mahasiswa', 'kampus']
D3: ['temu', 'dokumen', 'inverted', 'index']
D4: ['mahasiswa', 'akses', 'dokumen', 'sistem', 'temu']


## 4. Pembangunan Inverted Index

Setiap term dipetakan ke himpunan ID dokumen yang memuatnya (**posting list**).

$$\text{index}[t] = \{ d \mid t \in d \}$$

In [4]:
def build_inverted_index(tokenized):
    index = {}
    for doc_id, tokens in tokenized.items():
        for term in set(tokens):
            index.setdefault(term, set()).add(doc_id)
    return dict(sorted(index.items()))

inverted_index = build_inverted_index(tokenized)
print('Inverted Index:')
for term, postings in inverted_index.items():
    print(f'  {term:<12} -> {sorted(postings)}')

Inverted Index:
  ajar         -> ['D1']
  akses        -> ['D4']
  data         -> ['D2']
  dokumen      -> ['D3', 'D4']
  index        -> ['D3']
  informasi    -> ['D1', 'D2']
  inverted     -> ['D3']
  kampus       -> ['D2']
  kelola       -> ['D2']
  mahasiswa    -> ['D1', 'D2', 'D4']
  sistem       -> ['D1', 'D2', 'D4']
  temu         -> ['D1', 'D3', 'D4']


## 5. Eksekusi Boolean Query

- **AND** → irisan posting list (`A ∩ B`)
- **OR**  → gabungan posting list (`A ∪ B`)
- **NOT** → komplemen terhadap seluruh dokumen (`U \ A`)

In [5]:
ALL_DOCS = set(documents.keys())

def postings(term):
    return inverted_index.get(stemmer.stem(term.lower()), set())

def AND(a, b): return a & b
def OR(a, b):  return a | b
def NOT(a):    return ALL_DOCS - a

queries = [
    ('mahasiswa AND sistem',              AND(postings('mahasiswa'), postings('sistem'))),
    ('informasi OR dokumen',              OR(postings('informasi'), postings('dokumen'))),
    ('mahasiswa AND NOT dokumen',         AND(postings('mahasiswa'), NOT(postings('dokumen')))),
    ('(sistem OR dokumen) AND mahasiswa',
        AND(OR(postings('sistem'), postings('dokumen')), postings('mahasiswa'))),
]

for q, result in queries:
    print(f'{q:<40} -> {sorted(result) if result else "(tidak ada)"}')

mahasiswa AND sistem                     -> ['D1', 'D2', 'D4']
informasi OR dokumen                     -> ['D1', 'D2', 'D3', 'D4']
mahasiswa AND NOT dokumen                -> ['D1', 'D2']
(sistem OR dokumen) AND mahasiswa        -> ['D1', 'D2', 'D4']


## 6. Ringkasan

- **Inverted index** memetakan setiap term ke daftar dokumen, mempercepat pencarian dibanding *forward index*.
- Operasi boolean menjadi operasi himpunan sederhana pada posting list.
- Stopword & stemming menormalkan term sehingga query *mahasiswa* dan *mahasiswanya* dianggap sama.